# 🛒 Understanding the Online Retail Dataset

This notebook helps you explore the sample data generated for **RevenueOS**. We'll load the data using our own internal pipeline (`src.data.loader`) to see exactly what the machine sees.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px

# Add project root to Python path so we can import our modules
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data.loader import load_csv, get_dataframe_profile
from src.preprocessing.cleaner import run_cleaning_pipeline

print("✅ Environment ready!")

✅ Environment ready!


## 1. Load the Data
Let's load the CSV file using our custom `load_csv` function, which automatically handles encoding (useful for special characters like `£`) and parses dates.

In [2]:
data_path = project_root / "data" / "sample" / "online_retail_sample.csv"
df_raw = load_csv(data_path)

# Let's look at a quick profile
profile = get_dataframe_profile(df_raw)
print(f"Loaded {profile['rows']} rows and {profile['columns']} columns.")
print(f"Memory usage: {profile['memory_mb']} MB")

# Display first 5 rows
df_raw.head()

Loaded 5000 rows and 8 columns.
Memory usage: 1.34 MB


/Users/ariel/AIprojects/AI_REVENUE/src/data/loader.py:115: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], errors="coerce")
/Users/ariel/AIprojects/AI_REVENUE/src/data/loader.py:115: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], errors="coerce")
/Users/ariel/AIprojects/AI_REVENUE/src/data/loader.py:115: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], errors="coerce")
/Users/ariel/AIprojects/AI_REVENUE/src/data/loader.py:115: UserWarning: Could not infer format, so each el

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,C515795,22633,HAND WARMER UNION JACK,-7,2023-01-01 02:55:52.540205,2.13,12820.0,United Kingdom
1,500860,21078,SET/20 STRAWBERRY PAPER NAPKINS,15,2023-01-01 03:11:32.751655,0.99,13343.0,United Kingdom
2,576820,22752,SET 7 BABUSHKA NESTING BOXES,23,2023-01-01 04:24:50.357339,7.56,14295.0,United Kingdom
3,C554886,21078,SET/20 STRAWBERRY PAPER NAPKINS,-5,2023-01-01 05:13:13.034839,1.10,12003.0,United Kingdom
4,506265,21730,GLASS STAR FROSTED T-LIGHT HOLDER,13,2023-01-01 05:17:50.963172,4.25,14560.0,United Kingdom


### Column Descriptions:
- **InvoiceNo:** A 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation.
- **StockCode:** A 5-digit number uniquely assigned to each distinct product.
- **Description:** Product (item) name.
- **Quantity:** The quantities of each product per transaction. (Negative for cancellations).
- **InvoiceDate:** The day and time when a transaction was generated.
- **UnitPrice:** Product price per unit in sterling (£).
- **CustomerID:** A 5-digit number uniquely assigned to each customer.
- **Country:** The name of the country where a customer resides.

## 2. Clean the Data
In real life, data is messy. People return items (negative quantities), or checkout as guests (missing CustomerID). Our `run_cleaning_pipeline` fixes this.

In [3]:
print(f"Rows before cleaning: {len(df_raw)}")
df_clean = run_cleaning_pipeline(df_raw.copy())
print(f"Rows after cleaning: {len(df_clean)}")
print(f"Removed {len(df_raw) - len(df_clean)} noisy rows (cancellations, zero-price items, etc.)")

Rows before cleaning: 5000
Rows after cleaning: 5000
Removed 0 noisy rows (cancellations, zero-price items, etc.)


## 3. Basic Exploration (EDA)
What are the top selling countries? What's the distribution of order prices?

In [6]:
# Let's calculate total revenue per row (using lowercase column names from cleaner)
df_clean['revenue'] = df_clean['quantity'] * df_clean['unit_price']

# Top 5 Countries by Revenue
top_countries = df_clean.groupby('country')['revenue'].sum().sort_values(ascending=False).head(5)
print("Top 5 Countries by Revenue:")
print(top_countries)


KeyError: 'unit_price'

In [7]:
# Visualizing Daily Revenue (using lowercase invoice_date)
daily_revenue = df_clean.groupby(df_clean['invoice_date'].dt.date)['revenue'].sum().reset_index()
fig = px.line(
    daily_revenue, 
    x='invoice_date', 
    y='revenue', 
    title='Daily Revenue Over Time',
    template='plotly_dark'
)
fig.show()


KeyError: 'invoice_date'